In [1]:
from pathlib import Path

import duckdb
import pandas as pd

ROOT = Path(r"C:\Users\jeetd\Downloads\Databricks_proj\supplychainiq\supplychainiq")
RAW = ROOT / "data" / "raw"
sales  = (RAW / "m5" / "sales_train_evaluation.csv").as_posix()
cal    = (RAW / "m5" / "calendar.csv").as_posix()
prices = (RAW / "m5" / "sell_prices.csv").as_posix()
dataco = (RAW / "dataco" / "DataCoSupplyChainDataset.csv").as_posix()

con = duckdb.connect()

In [2]:
dataco_df = pd.read_csv(dataco, encoding="latin-1")
print(dataco_df.shape)

(180519, 53)


In [3]:
dataco_df["delay_days"] = (
    dataco_df["Days for shipping (real)"] - dataco_df["Days for shipment (scheduled)"]
)
pd.crosstab(dataco_df["delay_days"] > 0, dataco_df["Delivery Status"])

Delivery Status,Advance shipping,Late delivery,Shipping canceled,Shipping on time
delay_days,,,,
False,41592,0,3331,32196
True,0,98977,4423,0


In [4]:
delivered = dataco_df[dataco_df["Delivery Status"] != "Shipping canceled"].copy()
pd.crosstab(delivered["Delivery Status"], delivered["delay_days"])

delay_days,-2,-1,0,1,2,3,4
Delivery Status,,,,,,,
Advance shipping,20873,20719,0,0,0,0,0
Late delivery,0,0,0,57957,27551,6772,6697
Shipping on time,0,0,32196,0,0,0,0


In [5]:
print("rows:", len(delivered), "| unique orders:", delivered["Order Id"].nunique())

delivered.groupby("Order Id")["delay_days"].nunique().value_counts()

rows: 172765 | unique orders: 62897


delay_days
1    62897
Name: count, dtype: int64

In [6]:
orders = delivered.drop_duplicates("Order Id")

print(orders["delay_days"].describe())
print("median:", orders["delay_days"].median())
print("p90:", orders["delay_days"].quantile(0.9))
print("late share:", round((orders["delay_days"] > 0).mean(), 3))

count    62897.000000
mean         0.565893
std          1.493803
min         -2.000000
25%          0.000000
50%          1.000000
75%          1.000000
max          4.000000
Name: delay_days, dtype: float64
median: 1.0
p90: 2.0
late share: 0.573


In [7]:
pd.crosstab(orders["Shipping Mode"], orders["delay_days"], normalize="index").round(3)

delay_days,-2,-1,0,1,2,3,4
Shipping Mode,,,,,,,
First Class,0.000,0.0,0.000,1.000,0.000,0.0,0.0
Same Day,0.000,0.0,0.516,0.484,0.000,0.0,0.0
Second Class,0.000,0.0,0.200,0.198,0.202,0.2,0.2
Standard Class,0.202,0.2,0.200,0.198,0.200,0.0,0.0


In [8]:
orders.groupby("Shipping Mode")["Days for shipment (scheduled)"].value_counts()

Shipping Mode   Days for shipment (scheduled)
First Class     1                                 9602
Same Day        0                                 3407
Second Class    2                                12256
Standard Class  4                                37632
Name: count, dtype: int64

In [9]:
con.sql(f"""
    SELECT COUNT(*)                 AS item_store_rows,
           COUNT(DISTINCT item_id)  AS items,
           COUNT(DISTINCT store_id) AS stores,
           COUNT(DISTINCT state_id) AS states,
           COUNT(DISTINCT cat_id)   AS categories,
           COUNT(DISTINCT dept_id)  AS departments
    FROM read_csv_auto('{sales}')
""").df()

,item_store_rows,items,stores,states,categories,departments
0,30490,3049,10,3,3,7


In [10]:
# Date range covered by the sales columns d_1 ... d_1941
con.sql(f"""
    SELECT MIN(date) AS first_day, MAX(date) AS last_day, COUNT(*) AS days
    FROM read_csv_auto('{cal}')
    WHERE CAST(SUBSTR(d, 3) AS INTEGER) <= 1941
""").df()

,first_day,last_day,days
0,2011-01-29,2016-05-22,1941


In [11]:
WINDOW_START = "2014-05-23"   # last 2 years if the data ends 2016-05-22; check Cell 9 first

In [12]:
con.sql(f"""
    CREATE OR REPLACE TABLE sales_long AS
    SELECT u.item_id, u.dept_id, u.cat_id, u.store_id, u.state_id,
           c.date, c.wm_yr_wk, u.units
    FROM (
        UNPIVOT (SELECT * FROM read_csv_auto('{sales}'))
        ON COLUMNS('^d_') INTO NAME d VALUE units
    ) u
    JOIN read_csv_auto('{cal}') c ON u.d = c.d
    WHERE c.date >= DATE '{WINDOW_START}'
""")
con.sql("SELECT COUNT(*) AS rows FROM sales_long").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows
0,22288190


In [13]:
con.sql("""
    SELECT ROUND(AVG(CASE WHEN units = 0 THEN 1 ELSE 0 END), 3) AS zero_share
    FROM sales_long
""").df()

,zero_share
0,0.614


In [14]:
con.sql(f"""
    WITH win AS (
        SELECT MIN(wm_yr_wk) AS wk FROM read_csv_auto('{cal}') WHERE date >= DATE '{WINDOW_START}'
    ),
    per_item AS (
        SELECT store_id, item_id, COUNT(DISTINCT sell_price) AS n_prices
        FROM read_csv_auto('{prices}'), win
        WHERE wm_yr_wk >= win.wk
        GROUP BY store_id, item_id
    )
    SELECT COUNT(*) AS item_stores,
           ROUND(AVG(CASE WHEN n_prices > 1 THEN 1 ELSE 0 END), 3) AS share_with_price_change,
           MAX(n_prices) AS max_distinct_prices
    FROM per_item
""").df()

,item_stores,share_with_price_change,max_distinct_prices
0,30490,0.494,10


In [15]:
con.sql(f"""
    SELECT event_type_1, COUNT(*) AS days
    FROM read_csv_auto('{cal}')
    WHERE date >= DATE '{WINDOW_START}' AND CAST(SUBSTR(d, 3) AS INTEGER) <= 1941
      AND event_type_1 IS NOT NULL
    GROUP BY 1 ORDER BY 2 DESC
""").df()

,event_type_1,days
0,Religious,20
1,National,20
2,Cultural,13
3,Sporting,6


In [16]:
con.sql(f"""
    SELECT SUM(snap_CA) AS snap_CA, SUM(snap_TX) AS snap_TX, SUM(snap_WI) AS snap_WI
    FROM read_csv_auto('{cal}')
    WHERE date >= DATE '{WINDOW_START}' AND CAST(SUBSTR(d, 3) AS INTEGER) <= 1941
""").df()

,snap_CA,snap_TX,snap_WI
0,240.0,240.0,240.0
